# Data exploration

Checks on the input files as they are, before any cleaning. Each check ends with **R:**, the result.

- R1, Q1, ... are the rules and open questions in [`docs/design.md`](../docs/design.md).
- Issues found go to [`docs/data-issues.md`](../docs/data-issues.md) (D1, D2, ...).

## Setup

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data")

# Every file is read as text, so nothing is changed on the way in:
# spaces at the end of a name or odd values stay visible.
ledger = pd.read_csv(DATA / "ledger.csv", dtype=str)
entities = pd.read_csv(DATA / "entities.csv", dtype=str)
accounts = pd.read_csv(DATA / "accounts.csv", dtype=str)
cost_centers = pd.read_csv(DATA / "cost_centers.csv", dtype=str)
pools = pd.read_csv(DATA / "pools.csv", dtype=str)
headcount_games = pd.read_csv(DATA / "headcount_games.csv", dtype=str)
headcount_employees = pd.read_csv(DATA / "headcount_employees.csv", dtype=str)
game_mapping = pd.read_csv(DATA / "game_mapping.csv", dtype=str)

# The grain of each table, from the BRIEF's "One row per" column:
# the columns that should be unique together.
grain = {
    "ledger": (ledger, ["line_id"]),
    "entities": (entities, ["entity"]),
    "accounts": (accounts, ["account"]),
    "cost_centers": (cost_centers, ["cost_center"]),
    "pools": (pools, ["pool_id", "game"]),
    "headcount_games": (headcount_games, ["month", "game"]),
    "headcount_employees": (headcount_employees, ["month", "employee_id"]),
    "game_mapping": (game_mapping, ["game_group", "start_date"]),
}

## Each table on its own

### 1. Is every table unique at the grain the BRIEF states? (Q2)

In [2]:
# For each table: how many rows repeat the key columns?
rows = []
for name, (table, key) in grain.items():
    duplicated = table.duplicated(subset=key).sum()
    rows.append({"table": name, "rows": len(table), "key": key, "duplicated keys": duplicated})

pd.DataFrame(rows)

,table,rows,key,duplicated keys
0,ledger,646,[line_id],0
1,entities,5,[entity],0
2,accounts,9,[account],0
3,cost_centers,22,[cost_center],0
4,pools,12,"[pool_id, game]",0
5,headcount_games,26,"[month, game]",0
6,headcount_employees,368,"[month, employee_id]",0
7,game_mapping,16,"[game_group, start_date]",0


**R:** no table has a duplicated key, so no join will multiply rows.

### 2. Which columns have blanks or text with spaces around it? (Q1, Q8)

In [3]:
# For each column: how many values are blank, and how many have spaces at the start or end?
rows = []
for name, (table, key) in grain.items():
    for column in table.columns:
        values = table[column].dropna()
        blank = table[column].isna().sum()
        untrimmed = (values != values.str.strip()).sum()
        if blank > 0 or untrimmed > 0:
            rows.append({"table": name, "column": column, "blank": blank, "untrimmed": untrimmed})

pd.DataFrame(rows)

,table,column,blank,untrimmed
0,cost_centers,game,13,0
1,cost_centers,pool_id,18,0
2,game_mapping,game_group,0,1
3,game_mapping,bearing_group,10,0


In [4]:
print("rows with a blank bearing_group:", sorted(game_mapping.loc[game_mapping["bearing_group"].isna(), "game_group"].str.strip()))

rows with a blank bearing_group: ['Kingdom Clash', 'Legacy Arcade', 'Moon Miner', 'Ocean Tales', 'Pixel Farm', 'Puzzle Pals', 'Puzzle Pals', 'Rune Quest', 'Star Racer', 'Tiny Chef']


**R:**
- The blanks are expected: `game` and `pool_id` in cost centers (a cost center has one, the other or neither) and
  `bearing_group` in the mapping (filled only on bucket rows).
- One value in `game_mapping.game_group` has a space around it. Check 8 shows which one.
- The 10 blank bearing groups are exactly the 10 game rows, so every bucket row has one.

### 3. Are amounts and FTE valid numbers, and is every FTE above zero? (R1, R5)

In [5]:
# Numbers go in new columns; the original text stays as it was.
# errors="coerce" turns anything that is not a number into a blank, so it can be counted.
ledger["amount"] = pd.to_numeric(ledger["amount_usd"], errors="coerce")
headcount_games["fte_number"] = pd.to_numeric(headcount_games["fte"], errors="coerce")

pd.Series({
    "amount is not a number": ledger["amount"].isna().sum(),
    "amount = 0": (ledger["amount"] == 0).sum(),
    "amount < 0 (credits)": (ledger["amount"] < 0).sum(),
    "fte is not a number": headcount_games["fte_number"].isna().sum(),
    "fte <= 0": (headcount_games["fte_number"] <= 0).sum(),
})

amount is not a number    0
amount = 0                3
amount < 0 (credits)      2
fte is not a number       0
fte <= 0                  0
dtype: int64

**R:**
- All amounts and FTE are numbers, and every FTE is above zero.
- 3 lines have amount 0: out of scope (R1).
- 2 lines are credits.

### 4. What are the control totals? (R11)

Ledger lines and dollars per month, before anything is excluded. The reconciliation starts from these.

In [6]:
# Lines and dollars per month, plus a total row
control = ledger.groupby("month")["amount"].agg(lines="count", amount_usd="sum")
control.loc["total"] = control.sum()
control.round(2)

,lines,amount_usd
month,,
2026-01,210,2199715.43
2026-02,220,2208574.52
2026-03,216,2229005.81
total,646,6637295.76


**R:** 646 lines and 6,637,295.76 USD over three months (2026-01 to 2026-03).

### 5. How many lines and dollars fall out of scope, and why? (R1)

In [7]:
# Label every line with its scope reason, then add up lines and dollars per reason
scope = ledger.merge(accounts[["account", "account_group"]], on="account", how="left")

scope["reason"] = "in scope"
scope.loc[scope["account_group"] == "Depreciation", "reason"] = "depreciation"
scope.loc[scope["amount"] == 0, "reason"] = "zero amount"   # a zero line is out of scope whatever its account

in_scope = scope[scope["reason"] == "in scope"]
print("credits among in-scope lines:", (in_scope["amount"] < 0).sum())

scope.groupby("reason")["amount"].agg(lines="count", amount_usd="sum").round(2)

credits among in-scope lines: 2


,lines,amount_usd
reason,,
depreciation,12,98044.91
in scope,631,6539250.85
zero amount,3,0.00


**R:**
- Out of scope: 12 Depreciation lines (98,044.91) and 3 zero lines.
- In scope: 631 lines, 6,539,250.85.
- The three add up to the control total of check 4.
- 2 credits are in scope (Q6).
- A zero line is out of scope as "zero amount" even if its account is Depreciation.

### 6. Is every cost center set up as its department requires? (R3, Q1)

In [8]:
# Count cost centers per department by whether they have a game and/or a pool
setup = cost_centers.copy()
setup["has game"] = setup["game"].notna()
setup["has pool"] = setup["pool_id"].notna()

setup.groupby(["department", "has game", "has pool"]).size().reset_index(name="cost centers")

,department,has game,has pool,cost centers
0,Engineering,False,False,2
1,Live Games,False,True,2
2,Live Games,True,False,4
3,Marketing,False,False,2
4,New Games,False,True,2
5,New Games,True,False,5
6,Operations,False,False,3
7,People,False,False,2


**R:** yes.
- Live Games and New Games cost centers have a game or a pool, never both.
- Engineering, Marketing, People and Operations cost centers have neither, as indirect costs should.

### 7. Does every ledger line find its entity, account and cost center? (R2, R3, Q1)

In [9]:
# For each key: ledger values that don't exist in the reference table, with their lines and dollars
rows = []
for column, reference in [("entity", entities), ("account", accounts), ("cost_center", cost_centers)]:
    not_found = ~ledger[column].isin(reference[column])
    rows.append({
        "ledger column": column,
        "values not found": sorted(ledger.loc[not_found, column].unique()),
        "lines": not_found.sum(),
        "amount_usd": round(ledger.loc[not_found, "amount"].sum(), 2),
    })

pd.DataFrame(rows)

,ledger column,values not found,lines,amount_usd
0,entity,[],0,0.00
1,account,[],0,0.00
2,cost_center,[3999],1,17017.98


In [10]:
# The ledger lines whose cost center is not in cost_centers.csv
ledger[~ledger["cost_center"].isin(cost_centers["cost_center"])]

,line_id,month,entity,account,cost_center,amount_usd,amount
168,L00169,2026-02,LGB,6410,3999,17017.98,17017.98


**R:**
- Every line finds its entity and account.
- One line does not find its cost center: `L00169` (2026-02, LGB) on `3999`, 17,017.98. Without a department it has
  no path, so it stays unallocated with a reason (Q1).

### 8. Do game and pool names match across cost centers, pools, headcount and mapping? (R4, R5, R7, Q8)

In [11]:
# Game names used in each table that the mapping doesn't have
mapping_names = set(game_mapping["game_group"])

rows = []
for source, names in [
    ("cost_centers.game", cost_centers["game"]),
    ("pools.game", pools["game"]),
    ("headcount_games.game", headcount_games["game"]),
]:
    rows.append({"names from": source, "not found in game_mapping": sorted(set(names.dropna()) - mapping_names)})

pd.DataFrame(rows)

,names from,not found in game_mapping
0,cost_centers.game,[Star Racer]
1,pools.game,[Star Racer]
2,headcount_games.game,[Star Racer]


In [12]:
print("pool games with no FTE row:", sorted(set(pools["game"]) - set(headcount_games["game"])))
print("cost center pools not in pools.csv:", sorted(set(cost_centers["pool_id"].dropna()) - set(pools["pool_id"])))

# repr() shows the hidden space at the end of a name
untrimmed = game_mapping["game_group"] != game_mapping["game_group"].str.strip()
game_mapping.loc[untrimmed, "game_group"].map(repr)

pool games with no FTE row: []
cost center pools not in pools.csv: []


0    'Star Racer '
Name: game_group, dtype: str

**R:**
- Pools and FTE rows match.
- The mapping writes `"Star Racer "` with a space at the end, so it matches no cost center, pool or headcount row.
  Without a fix, every Star Racer line would lose its side. The pipeline trims names and logs the fix (Q8).

### 9. Does every employee bucket have a mapping row? (R8, R9, Q1)

In [13]:
employees = headcount_employees.merge(entities[["entity", "entity_group"]], on="entity", how="left")
print("employees whose entity is not in entities.csv:", employees["entity_group"].isna().sum())

# Bucket (R8): first word of the category + "_" + entity group, in lower case. "Live Games" + "BR" -> "live_br"
employees["bucket"] = (
    employees["category"].str.split().str[0].str.lower() + "_" + employees["entity_group"].str.lower()
)

buckets = employees.groupby(["bucket", "month"]).size().unstack("month", fill_value=0)
buckets["in mapping"] = buckets.index.isin(game_mapping["game_group"])
buckets

employees whose entity is not in entities.csv: 0


month,2026-01,2026-02,2026-03,in mapping
bucket,,,,
live_br,70,71,70,True
live_fi,0,0,2,False
new_br,25,25,26,True
new_fi,8,8,8,True
new_others,5,5,5,True
new_us,12,14,14,True


**R:**
- Every employee has a known entity.
- Five buckets have a mapping row. `live_fi` does not: Finland has 2 Live Games employees in 2026-03 and none before.
- In March, their share of the indirect costs has no bearing group or side, so it stays unallocated with a reason (Q1).

### 10. Does exactly one mapping row apply to each game and bucket in each month? (R10, Q3)

In [14]:
# Dates must be yyyy-MM-dd text, or comparing them as text below is wrong
for column in ["start_date", "end_date"]:
    not_valid = ~game_mapping[column].fillna("").str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    print(f"{column} not in yyyy-MM-dd:", not_valid.sum())

print(game_mapping[game_mapping["game_group"] == "Puzzle Pals"].to_string(index=False))

# For each month: how many mapping rows apply to each game or bucket on the month's first day
coverage = pd.DataFrame(index=game_mapping["game_group"].unique())

for month in sorted(ledger["month"].unique()):
    first_day = month + "-01"
    # Dates are yyyy-MM-dd text, which compares like a date; "9999-12-31" means no end date.
    applies = (game_mapping["start_date"] <= first_day) & (first_day <= game_mapping["end_date"])
    coverage[month] = game_mapping.loc[applies, "game_group"].value_counts()

coverage.fillna(0).astype(int)

start_date not in yyyy-MM-dd: 0
end_date not in yyyy-MM-dd: 0
 game_group  category bearing_group     side start_date   end_date
Puzzle Pals New Games           NaN BR Games 2025-01-01 2026-02-28
Puzzle Pals New Games           NaN US Games 2026-03-01 9999-12-31


,2026-01,2026-02,2026-03
Star Racer,1,1,1
Kingdom Clash,1,1,1
Ocean Tales,1,1,1
Legacy Arcade,1,1,1
Puzzle Pals,1,1,1
Moon Miner,0,1,1
Pixel Farm,1,1,1
Rune Quest,1,1,1
Tiny Chef,1,1,1
live_br,1,1,1


In [15]:
# Moon Miner has no mapping row in January: does anything need it that month?
lines_with_cost_center = ledger.merge(cost_centers, on="cost_center")
moon_miner = lines_with_cost_center[lines_with_cost_center["game"] == "Moon Miner"]

print("Moon Miner ledger lines per month:")
print(moon_miner.groupby("month").size())
print("Moon Miner FTE per month:")
print(headcount_games.loc[headcount_games["game"] == "Moon Miner", ["month", "fte"]].to_string(index=False))

Moon Miner ledger lines per month:
month
2026-02    13
2026-03    12
dtype: int64
Moon Miner FTE per month:
  month fte
2026-02 8.5
2026-03  10


**R:**
- Every start_date and end_date is yyyy-MM-dd: 0 values fail in either column.
- No game or bucket has two rows in the same month, so Q3 does not happen.
- Puzzle Pals changes from BR Games to US Games on 2026-03-01; the dates pick one row per month.
- Moon Miner has no row in January. That month it has no lines on its own cost center and no FTE, so no pool gives it anything. Nothing is affected.

### 11. Do the open questions on By entity buckets and the quarter happen? (Q5, Q9)

In [16]:
# Bucket rows are the ones with a bearing group
bucket_rows = game_mapping[game_mapping["bearing_group"].notna()]

print("bucket rows with side 'By entity':", (bucket_rows["side"] == "By entity").sum())
print("ledger months:", sorted(ledger["month"].unique()))

bucket rows with side 'By entity': 0
ledger months: ['2026-01', '2026-02', '2026-03']


**R:**
- No bucket row says By entity, so Q5 does not happen.
- The ledger covers 2026-01 to 2026-03, the first quarter of 2026 (Q9).

## Findings

| Check | Finding | Decided by | Logged as |
|---|---|---|---|
| 7 | Cost center `3999` is not in `cost_centers.csv`: line `L00169`, 2026-02, LGB, 17,017.98 | Q1 | D1 |
| 2, 8 | `"Star Racer "` has a space at the end in `game_mapping.csv`, so it matches no cost center, pool or headcount row | Q8 | D2 |
| 9 | Bucket `live_fi` (2 LGF Live Games employees in 2026-03) has no mapping row | Q1 | D3 |
| 3, 5 | 12 Depreciation lines and 3 zero lines are out of scope | R1 | expected |
| 3, 5 | 2 credits stay in scope | Q6 | expected |
| 10 | Moon Miner has no mapping row in January, and no lines or FTE that month | R10 | expected |
| 11 | No bucket row says By entity; the ledger covers the first quarter of 2026 | Q5, Q9 | update in `docs/design.md` |